In [1]:
import pandas as pd
from pathlib import Path

# Localiser le projet
project_dir = Path.cwd()

if project_dir.name == "notebooks":
    project_dir = project_dir.parent

splits_dir = project_dir / "data" / "splits"

# Charger Train et Validation
train_df = pd.read_csv(splits_dir / "train.csv")
val_df = pd.read_csv(splits_dir / "validation.csv")

# Réunir les deux partitions
developpement_df = pd.concat(
    [train_df, val_df],
    ignore_index=True
)

# Séparer X et y
target = "MedHouseVal"

X_dev = developpement_df.drop(columns=[target])
y_dev = developpement_df[target]

print("Train :", train_df.shape)
print("Validation :", val_df.shape)
print("Train + Validation :", developpement_df.shape)

print("X_dev :", X_dev.shape)
print("y_dev :", y_dev.shape)

Train : (14448, 9)
Validation : (3096, 9)
Train + Validation : (17544, 9)
X_dev : (17544, 8)
y_dev : (17544,)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from catboost import CatBoostRegressor
from time import perf_counter

# Conserver l'ordre des 8 caractéristiques
feature_columns = X_dev.columns.tolist()

# 1. Recréer le modèle CatBoost optimisé
catboost_final = CatBoostRegressor(
    iterations=800,
    learning_rate=0.09638924565901015,
    depth=8,
    l2_leaf_reg=2.453554179189145,
    loss_function="RMSE",
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1
)

# 2. Construire le pipeline
pipeline_final = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", catboost_final)
])

# 3. Entraîner sur Train + Validation
debut = perf_counter()

pipeline_final.fit(X_dev, y_dev)

temps_final = perf_counter() - debut

print("Pipeline entraîné avec succès.")
print(f"Observations utilisées : {len(X_dev)}")
print(f"Nombre de variables : {len(feature_columns)}")
print(f"Temps d'entraînement : {temps_final:.2f} s")

Pipeline entraîné avec succès.
Observations utilisées : 17544
Nombre de variables : 8
Temps d'entraînement : 5.60 s


In [3]:
# Utiliser 3 observations de Train + Validation
exemples = X_dev.iloc[:3]

# Faire des prédictions avec le pipeline
predictions = pipeline_final.predict(exemples)

print("Prédictions brutes :")
print(predictions)

print("\nPrédictions en dollars :")
print(predictions * 100000)

print("\nÉtapes du pipeline :")
print(pipeline_final.named_steps.keys())

Prédictions brutes :
[2.16810251 1.51937219 1.39837103]

Prédictions en dollars :
[216810.25054607 151937.21870825 139837.10348328]

Étapes du pipeline :
dict_keys(['imputer', 'model'])


In [4]:
import pandas as pd

from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

# Chemins
reports_dir = project_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

rapport_test = reports_dir / "evaluation_finale_test.csv"

# Éviter une seconde évaluation accidentelle
if rapport_test.exists():
    raise RuntimeError(
        "L'évaluation finale existe déjà. "
        "Consulte reports/evaluation_finale_test.csv."
    )

# 1. Charger uniquement le Test réservé
test_df = pd.read_csv(
    splits_dir / "test.csv"
)

# 2. Séparer les variables et la cible
X_test = test_df[feature_columns]
y_test = test_df["MedHouseVal"]

# 3. Prédire avec le pipeline final
y_pred_test = pipeline_final.predict(X_test)

# 4. Calculer les métriques
mae_test = mean_absolute_error(
    y_test, y_pred_test
)

rmse_test = root_mean_squared_error(
    y_test, y_pred_test
)

r2_test = r2_score(
    y_test, y_pred_test
)

# 5. Convertir les erreurs en dollars
mae_test_usd = mae_test * 100000
rmse_test_usd = rmse_test * 100000

# 6. Construire le rapport final
evaluation_finale = pd.DataFrame([{
    "Modele": "CatBoost",
    "Observations_Test": len(X_test),
    "MAE": mae_test,
    "RMSE": rmse_test,
    "R2": r2_test,
    "MAE_USD": mae_test_usd,
    "RMSE_USD": rmse_test_usd
}])

display(evaluation_finale.round(4))

# 7. Sauvegarder les résultats
evaluation_finale.to_csv(
    rapport_test,
    index=False
)

print(f"\nMAE finale : {mae_test_usd:,.2f} $")
print(f"RMSE finale : {rmse_test_usd:,.2f} $")
print(f"R² final : {r2_test:.4f}")

print("\nÉvaluation finale sauvegardée.")

,Modele,Observations_Test,MAE,RMSE,R2,MAE_USD,RMSE_USD
0,CatBoost,3096,0.2864,0.4362,0.8548,28644.5235,43615.6723



MAE finale : 28,644.52 $
RMSE finale : 43,615.67 $
R² final : 0.8548

Évaluation finale sauvegardée.


In [5]:
import joblib
from pathlib import Path

# 1. Localiser le dossier models
models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# 2. Définir le chemin du fichier
model_path = models_dir / "pipeline_immobilier.joblib"

# 3. Sauvegarder le pipeline entraîné
joblib.dump(
    pipeline_final,
    model_path
)

print("Pipeline sauvegardé avec succès.")
print("Chemin :", model_path)
print("Taille :", round(model_path.stat().st_size / 1024**2, 2), "Mo")

Pipeline sauvegardé avec succès.
Chemin : c:\5IASD\DEVELOPPEMENT DES APPLICATION INTELLIGENTES\TP\TP2-PREDICTION-PRIX-IMMOBILIER\models\pipeline_immobilier.joblib
Taille : 3.21 Mo


In [6]:
import numpy as np

# 1. Recharger le pipeline depuis le disque
pipeline_recharge = joblib.load(model_path)

# 2. Choisir 5 observations du Train + Validation
exemples = X_dev.iloc[:5]

# 3. Prédictions du pipeline original
pred_originales = pipeline_final.predict(exemples)

# 4. Prédictions du pipeline rechargé
pred_rechargees = pipeline_recharge.predict(exemples)

# 5. Vérifier la cohérence
identiques = np.allclose(
    pred_originales,
    pred_rechargees,
    rtol=1e-12,
    atol=1e-12
)

print("Prédictions originales :")
print(pred_originales)

print("\nPrédictions après rechargement :")
print(pred_rechargees)

print("\nPrédictions identiques :", identiques)

assert identiques, "Erreur : les prédictions sont différentes."

print("\nVérification de sérialisation réussie.")

Prédictions originales :
[2.16810251 1.51937219 1.39837103 3.63731902 1.92381789]

Prédictions après rechargement :
[2.16810251 1.51937219 1.39837103 3.63731902 1.92381789]

Prédictions identiques : True

Vérification de sérialisation réussie.


In [7]:
import json
import platform
import sklearn
import catboost

metadata = {
    "nom_modele": "CatBoostRegressor",
    "type_probleme": "regression",
    "dataset": "California Housing",

    "variables_entree": feature_columns,
    "nombre_variables": len(feature_columns),

    "cible": "MedHouseVal",
    "unite_cible": "100000_USD",

    "observations_entrainement": len(X_dev),

    "hyperparametres": {
        "iterations": 800,
        "learning_rate": 0.09638924565901015,
        "depth": 8,
        "l2_leaf_reg": 2.453554179189145
    },

    "evaluation_test": {
        "MAE": float(mae_test),
        "RMSE": float(rmse_test),
        "R2": float(r2_test)
    },

    "versions": {
        "python": platform.python_version(),
        "scikit_learn": sklearn.__version__,
        "catboost": catboost.__version__
    }
}

metadata_path = models_dir / "pipeline_metadata.json"

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Métadonnées sauvegardées.")
print("Chemin :", metadata_path)

Métadonnées sauvegardées.
Chemin : c:\5IASD\DEVELOPPEMENT DES APPLICATION INTELLIGENTES\TP\TP2-PREDICTION-PRIX-IMMOBILIER\models\pipeline_metadata.json
